# M4A_03: Train Schedule Features for Commuter Prediction

**Track A: Binary Classification (15-Minute Availability)**

## 🎯 Learning Objectives
By the end of this notebook, you will:
- Understand the relationship between train arrivals and bike demand
- Create features based on train schedules
- Engineer "minutes until next train" features
- Identify peak vs off-peak train periods
- Combine train schedule with time-based features

## 🚆 Why Train Schedule Features Matter

**OV-fiets Context**: Bikes are located at train stations for last-mile commuting.  
**Key Insight**: Bike demand **spikes** when trains arrive.

**Patterns to Capture**:
- Minutes until next train arrival (proximity to demand spike)
- Number of trains in next 15-30 minutes
- Peak vs off-peak train frequency (rush hour vs late night)
- Train type (intercity vs local = different volumes)

## ⚠️ Data Note
This notebook uses **simulated train schedules** for demonstration.  
In a real project, you would:
- Access Dutch Railways (NS) API
- Use historical train schedules
- Account for delays and cancellations

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load Bike Availability Data

In [ ]:
# Load data (with temporal and weather features)
df = pd.read_csv('../../../data/raw/sample_bike_weather.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Add temporal features for train schedule matching
df['hour'] = df['timestamp'].dt.hour
df['minute'] = df['timestamp'].dt.minute
df['day_of_week'] = df['timestamp'].dt.dayofweek

print(f"Dataset shape: {df.shape}")
df.head()

## 2. Create Simulated Train Schedule

**Typical Dutch Train Patterns**:
- **Peak hours** (6-9 AM, 4-7 PM): Trains every 10-15 minutes
- **Off-peak** (10 AM-4 PM): Trains every 15-30 minutes
- **Evening/night**: Trains every 30-60 minutes
- **Weekends**: Slightly less frequent

In [ ]:
def generate_train_schedule(date, station_id, is_weekday=True):
    """
    Generate simulated train arrival times for a day.
    
    Parameters:
    - date: datetime.date object
    - station_id: station identifier
    - is_weekday: bool (more frequent trains on weekdays)
    
    Returns:
    - DataFrame with train arrival timestamps
    """
    train_times = []
    
    # Define intervals based on time of day (minutes between trains)
    schedule = [
        (0, 6, 60),      # Night: hourly
        (6, 9, 10 if is_weekday else 15),   # Morning rush: every 10-15 min
        (9, 16, 15 if is_weekday else 20),  # Midday: every 15-20 min
        (16, 19, 10 if is_weekday else 15), # Evening rush: every 10-15 min
        (19, 24, 30)     # Evening: every 30 min
    ]
    
    for start_hour, end_hour, interval_minutes in schedule:
        current_time = datetime.combine(date, datetime.min.time()) + timedelta(hours=start_hour)
        end_time = datetime.combine(date, datetime.min.time()) + timedelta(hours=end_hour)
        
        while current_time < end_time:
            train_times.append({
                'train_arrival': current_time,
                'station_id': station_id
            })
            current_time += timedelta(minutes=interval_minutes)
    
    return pd.DataFrame(train_times)

# Generate train schedule for sample data dates
# (In real scenario, you'd have actual train schedules)
dates = df['timestamp'].dt.date.unique()
stations = df['station_id'].unique() if 'station_id' in df.columns else ['station_1']

all_trains = []
for date in dates:
    for station in stations:
        is_weekday = pd.Timestamp(date).dayofweek < 5
        station_trains = generate_train_schedule(date, station, is_weekday)
        all_trains.append(station_trains)

train_schedule = pd.concat(all_trains, ignore_index=True)
print(f"Generated {len(train_schedule)} train arrivals")
train_schedule.head(20)

## 3. Calculate "Minutes Until Next Train" Feature

For each bike availability record, find the next train arrival and calculate time difference.

In [ ]:
def minutes_until_next_train(row, train_schedule):
    """
    Find minutes until next train arrival for a given timestamp.
    """
    current_time = row['timestamp']
    station = row.get('station_id', 'station_1')  # Default if no station_id
    
    # Filter trains for this station and future arrivals
    future_trains = train_schedule[
        (train_schedule['station_id'] == station) & 
        (train_schedule['train_arrival'] > current_time)
    ]
    
    if len(future_trains) == 0:
        return 60  # Default: 60 minutes if no trains found
    
    # Get next train
    next_train = future_trains['train_arrival'].min()
    minutes_diff = (next_train - current_time).total_seconds() / 60
    
    return minutes_diff

# Calculate minutes until next train for each record
# Note: This can be slow for large datasets - consider optimization for production
print("Calculating minutes until next train (this may take a moment)...")
df['minutes_to_next_train'] = df.apply(lambda row: minutes_until_next_train(row, train_schedule), axis=1)

print("\nMinutes to next train statistics:")
print(df['minutes_to_next_train'].describe())

## 4. Create Train Proximity Features

Binary indicators for train arrivals in different time windows.

In [ ]:
# Binary indicators: Is a train arriving soon?
df['train_in_5min'] = (df['minutes_to_next_train'] <= 5).astype(int)
df['train_in_10min'] = (df['minutes_to_next_train'] <= 10).astype(int)
df['train_in_15min'] = (df['minutes_to_next_train'] <= 15).astype(int)

# Inverse feature: Time since last train
# (Useful to capture "bikes returning" pattern after train arrivals)
df['minutes_since_last_train'] = 60 - df['minutes_to_next_train']  # Simplified approximation

print("Train proximity features:")
print(f"Records with train in 5 min: {df['train_in_5min'].sum()}")
print(f"Records with train in 10 min: {df['train_in_10min'].sum()}")
print(f"Records with train in 15 min: {df['train_in_15min'].sum()}")

## 5. Train Frequency Features

How many trains are expected in the next 15-30 minutes?

In [ ]:
def count_trains_in_window(row, train_schedule, window_minutes=15):
    """
    Count trains arriving in the next N minutes.
    """
    current_time = row['timestamp']
    end_time = current_time + timedelta(minutes=window_minutes)
    station = row.get('station_id', 'station_1')
    
    trains_in_window = train_schedule[
        (train_schedule['station_id'] == station) & 
        (train_schedule['train_arrival'] > current_time) &
        (train_schedule['train_arrival'] <= end_time)
    ]
    
    return len(trains_in_window)

# Count trains in next 15 and 30 minutes
print("Counting trains in time windows (this may take a moment)...")
df['trains_in_15min'] = df.apply(lambda row: count_trains_in_window(row, train_schedule, 15), axis=1)
df['trains_in_30min'] = df.apply(lambda row: count_trains_in_window(row, train_schedule, 30), axis=1)

print("\nTrain frequency statistics:")
print(df[['trains_in_15min', 'trains_in_30min']].describe())

## 6. Visualize Train Impact on Bike Availability

In [ ]:
# Plot 1: Bike availability vs minutes to next train
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
# Bin minutes to next train for better visualization
df['minutes_bin'] = pd.cut(df['minutes_to_next_train'], bins=[0, 5, 10, 15, 20, 30, 60])
avg_by_proximity = df.groupby('minutes_bin')['bikes_available'].mean()

plt.bar(range(len(avg_by_proximity)), avg_by_proximity.values, color='steelblue')
plt.xticks(range(len(avg_by_proximity)), avg_by_proximity.index, rotation=45)
plt.xlabel('Minutes to Next Train')
plt.ylabel('Average Bikes Available')
plt.title('Bike Availability vs Train Arrival Proximity')
plt.grid(axis='y', alpha=0.3)

# Plot 2: Train frequency impact
plt.subplot(1, 2, 2)
train_freq_impact = df.groupby('trains_in_15min')['bikes_available'].mean()
plt.bar(train_freq_impact.index, train_freq_impact.values, color='coral')
plt.xlabel('Number of Trains in Next 15 Minutes')
plt.ylabel('Average Bikes Available')
plt.title('Bike Availability vs Train Frequency')
plt.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 7. TODO: Combine Train Schedule with Rush Hour Features

**Task**: Create interaction features between train schedules and time of day.

**Ideas**:
1. `train_in_rush_hour`: Train arriving during morning/evening rush
2. `high_frequency_period`: More than 3 trains in next 30 minutes
3. `critical_moment`: Train in 5 min AND rush hour

These interaction features capture **compound effects**.

In [ ]:
# TODO: Create interaction features

# Add rush hour indicators first (from M4A_01)
df['is_morning_rush'] = ((df['hour'] >= 6) & (df['hour'] < 9)).astype(int)
df['is_evening_rush'] = ((df['hour'] >= 16) & (df['hour'] < 19)).astype(int)
df['is_rush_hour'] = ((df['is_morning_rush'] == 1) | (df['is_evening_rush'] == 1)).astype(int)

# Example interactions:
# df['train_in_rush_hour'] = (df['train_in_15min'] == 1) & (df['is_rush_hour'] == 1)
# df['high_frequency_period'] = (df['trains_in_30min'] >= 3).astype(int)
# df['critical_moment'] = (df['train_in_5min'] == 1) & (df['is_rush_hour'] == 1)

# Your custom interaction features here:
# ...

print("Interaction features created")

## 8. Summary: Train Schedule Features for Track A

In [ ]:
# List train schedule features
train_features = [
    'minutes_to_next_train',
    'train_in_5min', 'train_in_10min', 'train_in_15min',
    'trains_in_15min', 'trains_in_30min',
    # Add your custom features here
]

existing_train_features = [f for f in train_features if f in df.columns]

print(f"Train schedule features created: {len(existing_train_features)}")
print(existing_train_features)
print("\nFeature statistics:")
print(df[existing_train_features].describe())

# Save updated dataset
# df.to_csv('../../../data/processed/track_a_all_features.csv', index=False)
# print("\n✅ All Track A features saved to data/processed/track_a_all_features.csv")

## 🎯 Key Takeaways

1. **Train arrivals drive bike demand** at station-based bike-sharing systems
2. **Proximity features** (minutes to next train) capture demand spikes
3. **Frequency features** (trains in next N minutes) indicate busy periods
4. **Interaction features** (train × rush hour) capture compound effects
5. **Real-world implementation** would use actual NS train schedules

## 🔗 Next Steps
You now have **all Track A features**:
- ✅ Temporal features (hour, rush hour, cyclical encodings)
- ✅ Weather features (temperature, rain, wind)
- ✅ Train schedule features (proximity, frequency)

**Next**: Move to **Module 5 Track A** for classification modeling!

## 📚 Resources
- [Dutch Railways (NS) API](https://www.ns.nl/en/travel-information/ns-api)
- [Track A README](./README.md)
- [Use Case Comparison Guide](../../../docs/guides/use_case_comparison.md)